### 03 构建探针特异的未标记候选池

本 notebook 只整理普通阳性位点并为每个实验条件独立生成未标记候选池，不固定抽取负样本。对于每个探针和残基类型，只在已经观测到该探针修饰该类残基的蛋白中枚举其他同类残基；随后排除当前条件下该探针所有已知阳性及其上下游 ±5 aa，并排除蛋白最前两个和最后两个残基。其他条件中的阳性位点仅保留为审计标记，不参与当前条件候选池的过滤。训练时如何抽样由后续建模代码决定。

In [8]:
# Cell 1：导入库和设置参数
import os
from collections import defaultdict
import pandas as pd

DATA_DIR = os.path.join("outputs", "positive_data")
FASTA_DIR = os.path.join("outputs", "fasta")
DATASET_DIR = os.path.join("outputs", "dataset")
EXCLUSION_RADIUS = 5

dataset_files = {
    "Human_Lysate_100uM": os.path.join(DATA_DIR, "Human_Lysate_100uM.csv"),
    "Human_Lysate_1mM": os.path.join(DATA_DIR, "Human_Lysate_1mM.csv"),
    "SA59_Lysate_100uM": os.path.join(DATA_DIR, "SA59_Lysate_100uM.csv"),
    "SA59_Lysate_1mM": os.path.join(DATA_DIR, "SA59_Lysate_1mM.csv"),
    "SA59_Insitu_100uM": os.path.join(DATA_DIR, "SA59_Insitu_100uM.csv"),
}

dataset_layout = {
    "Human_Lysate_100uM": ("Human", "Lysate", "100uM"),
    "Human_Lysate_1mM": ("Human", "Lysate", "1mM"),
    "SA59_Lysate_100uM": ("SA59", "Lysate", "100uM"),
    "SA59_Lysate_1mM": ("SA59", "Lysate", "1mM"),
    "SA59_Insitu_100uM": ("SA59", "Insitu", "100uM"),
}

In [9]:
# Cell 2：读取只含 GN 的正向蛋白序列
human_proteins = pd.read_excel(os.path.join(FASTA_DIR, "Huamn.xlsx"))
sa59_proteins = pd.read_excel(os.path.join(FASTA_DIR, "SA59.xlsx"))
human_sequences = human_proteins.set_index("UNIPROTID")["SEQUENCE"].to_dict()
sa59_sequences = sa59_proteins.set_index("UNIPROTID")["SEQUENCE"].to_dict()

print(f"人类正向蛋白：{len(human_sequences):,}")
print(f"SA59正向蛋白：{len(sa59_sequences):,}")

人类正向蛋白：20,868
SA59正向蛋白：2,959


In [10]:
# Cell 3：读取阳性数据，并把同一探针的多次尝试合并为位点并集
raw_tables, positive_tables, probe_worksheets = {}, {}, {}
terminal_tables, missing_sequence_tables = [], []
positive_datasets_by_site = defaultdict(set)

for dataset_name, file_path in dataset_files.items():
    data = pd.read_csv(file_path)
    required = {"Worksheet", "Probe", "Concentration", "UNIPROT_ID", "Position", "Site_type"}
    if not required.issubset(data.columns):
        raise ValueError(f"{file_path} 缺少字段：{sorted(required - set(data.columns))}")
    data.insert(0, "Dataset_name", dataset_name)
    sequences = human_sequences if dataset_name.startswith("Human") else sa59_sequences
    missing_sequence_tables.append(data[~data["UNIPROT_ID"].isin(sequences)].copy())
    data = data[data["UNIPROT_ID"].isin(sequences)].copy()
    terminal_tables.append(data[data["Site_type"].isin(["N-terminal", "C-terminal"])].copy())
    ordinary = data[~data["Site_type"].isin(["N-terminal", "C-terminal"])].copy()
    ordinary["Position"] = ordinary["Position"].astype(int)
    ordinary["Sequence_residue"] = [sequences[uid][position - 1] if 1 <= position <= len(sequences[uid]) else None for uid, position in ordinary[["UNIPROT_ID", "Position"]].itertuples(index=False)]
    mismatch = ordinary[ordinary["Sequence_residue"] != ordinary["Site_type"]]
    if len(mismatch):
        raise ValueError(f"{dataset_name} 有 {len(mismatch)} 个阳性位点与 FASTA 序列不一致")
    raw_tables[dataset_name] = ordinary
    probe_worksheets[dataset_name] = ordinary.groupby("Probe")["Worksheet"].agg(lambda x: sorted(set(x))).to_dict()
    for probe, uid, position in ordinary[["Probe", "UNIPROT_ID", "Position"]].itertuples(index=False):
        positive_datasets_by_site[(probe, uid, int(position))].add(dataset_name)

for dataset_name, ordinary in raw_tables.items():
    site_key = ["Probe", "Concentration", "UNIPROT_ID", "Position", "Site_type"]
    positives = ordinary.groupby(site_key, as_index=False).agg(N_positive_experiments=("Worksheet", "nunique"), Source_worksheets=("Worksheet", lambda x: ";".join(sorted(set(x)))))
    positives.insert(0, "Dataset_name", dataset_name)
    positives["N_probe_experiments"] = positives["Probe"].map(lambda p: len(probe_worksheets[dataset_name][p]))
    positives["Checked_worksheets"] = positives["Probe"].map(lambda p: ";".join(probe_worksheets[dataset_name][p]))
    positives["Label"] = 1
    positives["Sample_type"] = "observed_positive"
    positive_tables[dataset_name] = positives
    print(f"{dataset_name}: 可用普通位点记录={len(ordinary):,}，合并后唯一阳性位点={len(positives):,}")

Human_Lysate_100uM: 可用普通位点记录=16,447，合并后唯一阳性位点=16,447
Human_Lysate_1mM: 可用普通位点记录=461，合并后唯一阳性位点=461
SA59_Lysate_100uM: 可用普通位点记录=41,991，合并后唯一阳性位点=41,991
SA59_Lysate_1mM: 可用普通位点记录=17,126，合并后唯一阳性位点=17,126
SA59_Insitu_100uM: 可用普通位点记录=6,503，合并后唯一阳性位点=6,503


In [11]:
# Cell 4：构建完整的探针特异候选池
def build_candidate_pool(dataset_name, positives):
    sequences = human_sequences if dataset_name.startswith("Human") else sa59_sequences
    current_positive_positions = (
        positives.groupby(["Probe", "UNIPROT_ID"])["Position"]
        .agg(lambda values: tuple(sorted(set(map(int, values)))))
        .to_dict()
    )
    pool_rows = []
    for (probe, concentration), probe_data in positives.groupby(["Probe", "Concentration"], sort=False):
        for residue, residue_positives in probe_data.groupby("Site_type", sort=False):
            eligible_proteins = sorted(residue_positives["UNIPROT_ID"].unique())
            for uid in eligible_proteins:
                sequence = sequences[uid]
                known_positions = current_positive_positions.get((probe, uid), ())
                blocked = {p for positive_position in known_positions for p in range(max(1, positive_position - EXCLUSION_RADIUS), min(len(sequence), positive_position + EXCLUSION_RADIUS) + 1)}
                terminal_guard = {p for p in (1, 2, len(sequence) - 1, len(sequence)) if 1 <= p <= len(sequence)}
                candidates = tuple(position for position, amino_acid in enumerate(sequence, start=1) if amino_acid == residue and position not in blocked and position not in terminal_guard)
                if candidates:
                    other_condition_positive_positions = tuple(
                        position for position in candidates
                        if positive_datasets_by_site.get((probe, uid, position), set()) - {dataset_name}
                    )
                    pool_rows.append({"Dataset_name": dataset_name, "Probe": probe, "Concentration": concentration, "UNIPROT_ID": uid, "Site_type": residue, "Known_positive_positions_current_condition": known_positions, "Positive_in_other_condition_positions": other_condition_positive_positions, "Excluded_terminal_guard_positions": tuple(sorted(terminal_guard)), "Candidate_positions": candidates, "N_candidates": len(candidates)})
    return pd.DataFrame(pool_rows)

In [12]:
# Cell 5：按物种、浓度和处理方式保存数据
candidate_pools, output_dirs = {}, {}
for dataset_name, positives in positive_tables.items():
    organism, treatment_dir, concentration_dir = dataset_layout[dataset_name]
    output_dir = os.path.join(DATASET_DIR, organism, treatment_dir, concentration_dir)
    os.makedirs(output_dir, exist_ok=True)
    output_dirs[dataset_name] = output_dir
    positives = positives.copy()
    positives["Sample_type"] = "observed_positive"
    candidate_pool = build_candidate_pool(dataset_name, positives)
    candidate_pools[dataset_name] = candidate_pool
    positives.to_csv(os.path.join(output_dir, "positive_sites.csv"), index=False, encoding="utf-8-sig")
    candidate_pool.to_pickle(os.path.join(output_dir, "candidate_pool.pkl.gz"), compression="gzip")
    print(f"{dataset_name}: 阳性={len(positives):,}，候选池位置={candidate_pool['N_candidates'].sum():,}")

excluded_terminal = pd.concat(terminal_tables, ignore_index=True)
excluded_missing_sequence = pd.concat(missing_sequence_tables, ignore_index=True)
for dataset_name, output_dir in output_dirs.items():
    excluded_terminal[excluded_terminal["Dataset_name"].eq(dataset_name)].to_csv(os.path.join(output_dir, "excluded_terminal_positives.csv"), index=False, encoding="utf-8-sig")
    excluded_missing_sequence[excluded_missing_sequence["Dataset_name"].eq(dataset_name)].to_csv(os.path.join(output_dir, "excluded_missing_sequence_positives.csv"), index=False, encoding="utf-8-sig")

Human_Lysate_100uM: 阳性=16,447，候选池位置=185,944
Human_Lysate_1mM: 阳性=461，候选池位置=8,535
SA59_Lysate_100uM: 阳性=41,991，候选池位置=249,746
SA59_Lysate_1mM: 阳性=17,126，候选池位置=103,859
SA59_Insitu_100uM: 阳性=6,503，候选池位置=53,660


In [ ]:
# Cell 6：最终检查和候选池统计
summary_rows, residue_summary_tables = [], []
for dataset_name, positives in positive_tables.items():
    sequences = human_sequences if dataset_name.startswith("Human") else sa59_sequences
    pool = candidate_pools[dataset_name]
    key = ["Probe", "Concentration", "UNIPROT_ID", "Position", "Site_type"]
    assert not positives.duplicated(key).any()
    assert not pool.duplicated(["Probe", "Concentration", "UNIPROT_ID", "Site_type"]).any()
    for row in pool.itertuples(index=False):
        assert not set(row.Candidate_positions) & set(row.Known_positive_positions_current_condition)
        assert all(abs(position - positive_position) > EXCLUSION_RADIUS for position in row.Candidate_positions for positive_position in row.Known_positive_positions_current_condition)
        expected_other_condition_positions = {position for position in row.Candidate_positions if positive_datasets_by_site.get((row.Probe, row.UNIPROT_ID, position), set()) - {dataset_name}}
        assert set(row.Positive_in_other_condition_positions) == expected_other_condition_positions
        assert not set(row.Candidate_positions) & set(row.Excluded_terminal_guard_positions)
        assert all(sequences[row.UNIPROT_ID][position - 1] == row.Site_type for position in row.Candidate_positions)
        assert all(2 < position < len(sequences[row.UNIPROT_ID]) - 1 for position in row.Candidate_positions)
    positive_by_type = positives.groupby(["Probe", "Concentration", "Site_type"]).size().rename("Positive_sites")
    pool_by_type = pool.groupby(["Probe", "Concentration", "Site_type"])["N_candidates"].sum().rename("Candidate_sites")
    residue_summary = pd.concat([positive_by_type, pool_by_type], axis=1).fillna(0).reset_index()
    residue_summary.insert(0, "Dataset", dataset_name)
    residue_summary_tables.append(residue_summary)
    summary_rows.append({"Dataset": dataset_name, "Positive_sites": len(positives), "Candidate_sites": int(pool["N_candidates"].sum()), "Candidate_groups": len(pool), "Retained_other_condition_positives": int(pool["Positive_in_other_condition_positions"].map(len).sum()), "Excluded_terminal": int(excluded_terminal["Dataset_name"].eq(dataset_name).sum()), "Excluded_missing_sequence": int(excluded_missing_sequence["Dataset_name"].eq(dataset_name).sum())})
summary = pd.DataFrame(summary_rows)
residue_summary = pd.concat(residue_summary_tables, ignore_index=True)
summary_dir = os.path.join(DATASET_DIR, "summary")
os.makedirs(summary_dir, exist_ok=True)
summary.to_csv(os.path.join(summary_dir, "candidate_pool_summary.csv"), index=False, encoding="utf-8-sig")
residue_summary.to_csv(os.path.join(summary_dir, "candidate_pool_by_residue.csv"), index=False, encoding="utf-8-sig")
summary

,Dataset,Positive_sites,Candidate_sites,Candidate_groups,Retained_other_condition_positives,Excluded_terminal,Excluded_missing_sequence
0,Human_Lysate_100uM,16447,185944,9142,0,150,0
1,Human_Lysate_1mM,461,8535,287,0,1,0
2,SA59_Lysate_100uM,41991,249746,19377,5549,1568,0
3,SA59_Lysate_1mM,17126,103859,6565,1723,977,0
4,SA59_Insitu_100uM,6503,53660,3286,4511,288,0


: 